In [1]:
!pip install decord

In [2]:
import os
import json
import time
from typing import List, Dict, Tuple

import numpy as np
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import CLIPModel, CLIPProcessor
from decord import VideoReader, cpu

In [3]:
#Cell 2 — mount Drive, set paths:

from google.colab import drive
drive.mount('/content/drive')

MSVD_ROOT = "/content/drive/Shareddrives/DATA 298A/DATA/MSVD"
MSRVTT_ROOT = "/content/drive/Shareddrives/DATA 298A/DATA/MSRVTT"

CHECKPOINT = "/content/drive/Shareddrives/DATA 298A/Models/CLIP/output_msvd_clip/best.pt"

print("MSVD_ROOT exists:", os.path.exists(MSVD_ROOT))
print("MSRVTT_ROOT exists:", os.path.exists(MSRVTT_ROOT))
print("CHECKPOINT exists:", os.path.exists(CHECKPOINT))

Mounted at /content/drive
MSVD_ROOT exists: True
MSRVTT_ROOT exists: True
CHECKPOINT exists: True


In [4]:
#Cell 3 — helper functions, model class, and checkpoint loader:
def load_json(path: str):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def sample_frame_indices(total_frames: int, num_frames: int) -> List[int]:
    if total_frames <= 0:
        raise ValueError("Video has no frames")
    return np.linspace(0, total_frames - 1, num_frames, dtype=int).tolist()


def load_video_frames(video_path: str, num_frames: int) -> List[Image.Image]:
    vr = VideoReader(video_path, ctx=cpu(0))
    idxs = sample_frame_indices(len(vr), num_frames)
    frames = vr.get_batch(idxs).asnumpy()
    return [Image.fromarray(frame).convert("RGB") for frame in frames]


def build_test_set_msvd(data_root: str) -> Tuple[List[Dict], List[Dict]]:
    test_file = os.path.join(data_root, "msvd_test.json")
    video_root = os.path.join(data_root, "raw_videos")

    if not os.path.exists(test_file):
        raise FileNotFoundError(f"Missing file: {test_file}")
    if not os.path.exists(video_root):
        raise FileNotFoundError(f"Missing folder: {video_root}")

    records = load_json(test_file)

    videos = {}
    queries = []

    for item in records:
        video_name = item["video"]
        video_id = item.get("video_id", video_name)
        video_path = os.path.join(video_root, video_name)

        if not os.path.exists(video_path):
            continue

        if video_id not in videos:
            videos[video_id] = {
                "video_id": video_id,
                "video_name": video_name,
                "video_path": video_path,
            }

        captions = item["caption"]
        if isinstance(captions, str):
            captions = [captions]

        for cap in captions:
            cap = " ".join(str(cap).strip().split())
            if cap:
                queries.append({
                    "video_id": video_id,
                    "caption": cap
                })

    return queries, list(videos.values())


def extract_tensor_features(output, kind="image"):
    if torch.is_tensor(output):
        return output
    if hasattr(output, "image_embeds") and output.image_embeds is not None:
        return output.image_embeds
    if hasattr(output, "text_embeds") and output.text_embeds is not None:
        return output.text_embeds
    if hasattr(output, "pooler_output") and output.pooler_output is not None:
        return output.pooler_output
    if hasattr(output, "last_hidden_state") and output.last_hidden_state is not None:
        return output.last_hidden_state[:, 0, :]
    if isinstance(output, (tuple, list)) and len(output) > 0:
        if torch.is_tensor(output[0]):
            return output[0]
    raise TypeError(f"Could not extract tensor features from {type(output)}")


class CLIPVideoTextModel(nn.Module):
    def __init__(self, model_name: str):
        super().__init__()
        self.clip = CLIPModel.from_pretrained(model_name)

    def encode_text(self, input_ids, attention_mask):
        try:
            text_features = self.clip.get_text_features(
                input_ids=input_ids,
                attention_mask=attention_mask
            )
        except Exception:
            out = self.clip.text_model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )
            text_features = extract_tensor_features(out, kind="text")

        text_features = extract_tensor_features(text_features, kind="text")
        return F.normalize(text_features, dim=-1)

    def encode_video(self, pixel_values):
        bsz, t, c, h, w = pixel_values.shape
        flat_pixels = pixel_values.reshape(bsz * t, c, h, w)

        try:
            frame_features = self.clip.get_image_features(pixel_values=flat_pixels)
        except Exception:
            out = self.clip.vision_model(pixel_values=flat_pixels)
            frame_features = extract_tensor_features(out, kind="image")

        frame_features = extract_tensor_features(frame_features, kind="image")
        frame_features = frame_features.reshape(bsz, t, -1)
        video_features = frame_features.mean(dim=1)

        return F.normalize(video_features, dim=-1)


def load_checkpoint(model: nn.Module, checkpoint_path: str):
    if not os.path.exists(checkpoint_path):
        raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")

    ckpt = torch.load(checkpoint_path, map_location="cpu")

    if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
        model.load_state_dict(ckpt["model_state_dict"], strict=False)
    else:
        model.load_state_dict(ckpt, strict=False)

    print(f"Loaded checkpoint from: {checkpoint_path}")


MODEL_NAME = "openai/clip-vit-base-patch32"
NUM_FRAMES = 8
TEXT_BATCH_SIZE = 64
VIDEO_BATCH_SIZE = 16
MAX_TEXT_LEN = 32

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

processor = CLIPProcessor.from_pretrained(MODEL_NAME)
model = CLIPVideoTextModel(MODEL_NAME).to(device)
load_checkpoint(model, CHECKPOINT)
model.eval()

Using device: cuda


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

model.safetensors: downloading bytes:           |  0.00B            

Loaded checkpoint from: /content/drive/Shareddrives/DATA 298A/Models/CLIP/output_msvd_clip/best.pt


CLIPVideoTextModel(
  (clip): CLIPModel(
    (text_model): CLIPTextModel(
      (embeddings): CLIPTextEmbeddings(
        (token_embedding): Embedding(49408, 512)
        (position_embedding): Embedding(77, 512)
      )
      (encoder): CLIPEncoder(
        (layers): ModuleList(
          (0-11): 12 x CLIPEncoderLayer(
            (self_attn): CLIPAttention(
              (k_proj): Linear(in_features=512, out_features=512, bias=True)
              (v_proj): Linear(in_features=512, out_features=512, bias=True)
              (q_proj): Linear(in_features=512, out_features=512, bias=True)
              (out_proj): Linear(in_features=512, out_features=512, bias=True)
            )
            (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
            (mlp): CLIPMLP(
              (activation_fn): QuickGELUActivation()
              (fc1): Linear(in_features=512, out_features=2048, bias=True)
              (fc2): Linear(in_features=2048, out_features=512, bias=True)
   

In [5]:
#Cell 4: run the actual MSVD evaluation
@torch.no_grad()
def encode_videos(model, processor, videos, device, num_frames, batch_size):
    all_embeds = []
    video_ids = []

    for start in tqdm(range(0, len(videos), batch_size), desc="Encoding videos"):
        batch = videos[start:start + batch_size]

        frames_nested = [load_video_frames(v["video_path"], num_frames) for v in batch]
        flat_images = [img for frames in frames_nested for img in frames]

        image_inputs = processor(images=flat_images, return_tensors="pt")
        pixel_values = image_inputs["pixel_values"]

        pixel_values = pixel_values.reshape(
            len(batch), num_frames, *pixel_values.shape[1:]
        ).to(device)

        video_embeds = model.encode_video(pixel_values)
        all_embeds.append(video_embeds.cpu())
        video_ids.extend([v["video_id"] for v in batch])

    return torch.cat(all_embeds, dim=0), video_ids


@torch.no_grad()
def encode_queries(model, processor, queries, device, max_text_len, batch_size):
    all_embeds = []
    gt_video_ids = []

    for start in tqdm(range(0, len(queries), batch_size), desc="Encoding texts"):
        batch = queries[start:start + batch_size]
        texts = [q["caption"] for q in batch]
        gt_video_ids.extend([q["video_id"] for q in batch])

        text_inputs = processor(
            text=texts,
            padding=True,
            truncation=True,
            max_length=max_text_len,
            return_tensors="pt"
        )

        text_embeds = model.encode_text(
            text_inputs["input_ids"].to(device),
            text_inputs["attention_mask"].to(device)
        )
        all_embeds.append(text_embeds.cpu())

    return torch.cat(all_embeds, dim=0), gt_video_ids


def compute_metrics(similarity: np.ndarray, gt_video_ids: List[str], video_ids: List[str]) -> Dict[str, float]:
    video_id_to_idx = {vid: i for i, vid in enumerate(video_ids)}
    gt_indices = np.array([video_id_to_idx[v] for v in gt_video_ids], dtype=np.int64)

    sorted_idx = np.argsort(-similarity, axis=1)
    ranks = []

    for i in range(len(gt_indices)):
        rank = int(np.where(sorted_idx[i] == gt_indices[i])[0][0]) + 1
        ranks.append(rank)

    ranks = np.array(ranks)

    return {
        "R@1": float(np.mean(ranks <= 1)),
        "R@5": float(np.mean(ranks <= 5)),
        "R@10": float(np.mean(ranks <= 10)),
        "MRR": float(np.mean(1.0 / ranks)),
        "MeanRank": float(np.mean(ranks)),
        "MedianRank": float(np.median(ranks)),
        "Top1Accuracy": float(np.mean(ranks == 1)),
        "Top5Accuracy": float(np.mean(ranks <= 5)),
    }


queries_msvd, videos_msvd = build_test_set_msvd(MSVD_ROOT)
print(f"MSVD — Queries: {len(queries_msvd)} | Videos: {len(videos_msvd)}")

start_time = time.time()
video_embeds_msvd, video_ids_msvd = encode_videos(model, processor, videos_msvd, device, NUM_FRAMES, VIDEO_BATCH_SIZE)
text_embeds_msvd, gt_ids_msvd = encode_queries(model, processor, queries_msvd, device, MAX_TEXT_LEN, TEXT_BATCH_SIZE)
encode_time_msvd = time.time() - start_time

similarity_msvd = (text_embeds_msvd @ video_embeds_msvd.T).numpy()
metrics_msvd = compute_metrics(similarity_msvd, gt_ids_msvd, video_ids_msvd)
metrics_msvd["avg_query_latency_sec"] = float(encode_time_msvd / max(len(queries_msvd), 1))

print("\nFine-tuned CLIP — MSVD Retrieval Metrics")
for k, v in metrics_msvd.items():
    print(f"{k}: {v}")

MSVD — Queries: 27763 | Videos: 670


Encoding texts: 100%|██████████| 434/434 [00:18<00:00, 23.90it/s]



Fine-tuned CLIP — MSVD Retrieval Metrics
R@1: 0.36638691784029104
R@5: 0.6792493606598711
R@10: 0.7866945214854303
MRR: 0.5066886064597899
MeanRank: 14.134351474984692
MedianRank: 2.0
Top1Accuracy: 0.36638691784029104
Top5Accuracy: 0.6792493606598711
avg_query_latency_sec: 0.03653409654261232


In [6]:
#cell5

msrvtt_test_path = os.path.join(MSRVTT_ROOT, "msrvtt_test_1k.json")
with open(msrvtt_test_path, "r", encoding="utf-8") as f:
    msrvtt_data = json.load(f)

print("Type of loaded data:", type(msrvtt_data))
print("Number of records:", len(msrvtt_data) if hasattr(msrvtt_data, '__len__') else "N/A")
print("\nFirst record (raw):")
print(msrvtt_data[0] if isinstance(msrvtt_data, list) else list(msrvtt_data.items())[0])

Type of loaded data: <class 'list'>
Number of records: 1000

First record (raw):
{'video_id': 'video7020', 'video': 'video7020.mp4', 'caption': 'a woman creating a fondant baby and flower', 'source': 'MSR-VTT', 'category': 10, 'url': 'https://www.youtube.com/watch?v=4KRMJNBjrEs', 'start time': 221.77, 'end time': 233.08, 'id': 7020}


In [7]:
#Cell 6 — MSR-VTT test set builder + run:

def build_test_set_msrvtt(data_root: str) -> Tuple[List[Dict], List[Dict]]:
    test_file = os.path.join(data_root, "msrvtt_test_1k.json")
    video_root = os.path.join(data_root, "raw_videos")

    if not os.path.exists(test_file):
        raise FileNotFoundError(f"Missing file: {test_file}")
    if not os.path.exists(video_root):
        raise FileNotFoundError(f"Missing folder: {video_root}")

    records = load_json(test_file)

    videos = {}
    queries = []

    for item in records:
        video_name = item["video"]
        video_id = item.get("video_id", video_name)
        video_path = os.path.join(video_root, video_name)

        if not os.path.exists(video_path):
            continue

        if video_id not in videos:
            videos[video_id] = {
                "video_id": video_id,
                "video_name": video_name,
                "video_path": video_path,
            }

        captions = item["caption"]
        if isinstance(captions, str):
            captions = [captions]

        for cap in captions:
            cap = " ".join(str(cap).strip().split())
            if cap:
                queries.append({
                    "video_id": video_id,
                    "caption": cap
                })

    return queries, list(videos.values())


queries_msrvtt, videos_msrvtt = build_test_set_msrvtt(MSRVTT_ROOT)
print(f"MSR-VTT — Queries: {len(queries_msrvtt)} | Videos found: {len(videos_msrvtt)}")
print("Sample query:", queries_msrvtt[0] if queries_msrvtt else "NONE")
print("Sample video:", videos_msrvtt[0] if videos_msrvtt else "NONE")

MSR-VTT — Queries: 1000 | Videos found: 1000
Sample query: {'video_id': 'video7020', 'caption': 'a woman creating a fondant baby and flower'}
Sample video: {'video_id': 'video7020', 'video_name': 'video7020.mp4', 'video_path': '/content/drive/Shareddrives/DATA 298A/DATA/MSRVTT/raw_videos/video7020.mp4'}


In [9]:
#Cell 7 — run the actual encoding + metrics for MSR-VTT

start_time = time.time()
video_embeds_msrvtt, video_ids_msrvtt = encode_videos(model, processor, videos_msrvtt, device, NUM_FRAMES, VIDEO_BATCH_SIZE)
text_embeds_msrvtt, gt_ids_msrvtt = encode_queries(model, processor, queries_msrvtt, device, MAX_TEXT_LEN, TEXT_BATCH_SIZE)
encode_time_msrvtt = time.time() - start_time

similarity_msrvtt = (text_embeds_msrvtt @ video_embeds_msrvtt.T).numpy()
metrics_msrvtt = compute_metrics(similarity_msrvtt, gt_ids_msrvtt, video_ids_msrvtt)
metrics_msrvtt["avg_query_latency_sec"] = float(encode_time_msrvtt / max(len(queries_msrvtt), 1))

print("\nFine-tuned CLIP — MSR-VTT Retrieval Metrics")
for k, v in metrics_msrvtt.items():
    print(f"{k}: {v}")

Encoding texts: 100%|██████████| 16/16 [00:00<00:00, 22.32it/s]


Fine-tuned CLIP — MSR-VTT Retrieval Metrics
R@1: 0.34
R@5: 0.584
R@10: 0.689
MRR: 0.4560530065702406
MeanRank: 27.761
MedianRank: 3.0
Top1Accuracy: 0.34
Top5Accuracy: 0.584
avg_query_latency_sec: 1.0233718147277833


In [11]:
#cell 8 - save both results
combined_results = {
    "model_type": "fine_tuned",
    "checkpoint": CHECKPOINT,
    "msvd": metrics_msvd,
    "msrvtt": metrics_msrvtt,
}

OUTPUT_JSON = "/content/clip_finetuned_results.json"
with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(combined_results, f, indent=2)

print(f"Saved to {OUTPUT_JSON}")

Saved to /content/clip_finetuned_results.json
